In [1]:
# Episodic Pivot / 20DMA Overlap strategy (daily + 3-minute)
# -----------------------------------------------------------------
# This cell follows AGENTS.md requirements exactly:
# - Inter-day data is loaded via functions from strategies.data.
# - Intra-day data is loaded via price_data_import functions.
# - Backtesting is done with backtrader.
#
# Strategy rules implemented:
# Entry flag (daily):
#   1) Gap up >= 8% from previous close to current open.
#   2) Daily close > 20DMA.
#   3) Flag remains active while daily close stays above 20DMA.
# Entry trigger (same date):
#   1) Daily overlap with 20DMA: High > 20DMA > Low.
#   2) 3-minute close < VWAP.
#   3) 3-minute EMA(5) crosses above EMA(9).
# Exit:
#   1) Daily close < 20DMA, OR
#   2) 3-minute close < (20DMA - 2 * ATR).
# Stop:
#   - Low of entry day (implemented as running day-low captured at entry time).

from __future__ import annotations

import backtrader as bt
import pandas as pd
from IPython.display import display

from strategies.data import add_market_data_to_syspath, load_daily_variables


# Make sibling `market_data` importable for intra-day pull functions.
add_market_data_to_syspath()

# AGENTS.md requires intra-day imports from price_data_import.
try:
    from market_data import price_data_import as _pdi
except ImportError:
    import price_data_import as _pdi


INTRADAY_IMPORTERS = {}
for _name in ("intraday_import", "nonconsecutive_intraday_import", "fragmented_intraday_import"):
    _func = getattr(_pdi, _name, None)
    if callable(_func):
        INTRADAY_IMPORTERS[_name] = _func

if not INTRADAY_IMPORTERS:
    raise ImportError(
        "Could not import intraday import functions from price_data_import. "
        "Expected at least one of: intraday_import, nonconsecutive_intraday_import, fragmented_intraday_import"
    )







In [2]:
# Multi-symbol extensions (modeled after example_multi_symbol output style)
# -------------------------------------------------------------------
# This cell adds:
# 1) TradeCaptureAnalyzer for closed-trade reporting.
# 2) Multi-symbol runner that aggregates per-symbol Backtrader runs.
# 3) Display helper that prints/displays summary tables similar to example_multi_symbol.ipynb.


class TradeCaptureAnalyzer(bt.Analyzer):
    """Collect closed-trade metadata for reporting."""

    def start(self) -> None:
        self.trades: list[dict] = []

    @staticmethod
    def _safe_history_value(trade, index: int, attr: str):
        try:
            return float(getattr(trade.history[index].event, attr))
        except Exception:
            return None

    def notify_trade(self, trade) -> None:
        if not trade.isclosed:
            return

        symbol = trade.data._name or "UNKNOWN"
        entry_dt = bt.num2date(trade.dtopen)
        exit_dt = bt.num2date(trade.dtclose)

        entry_price = self._safe_history_value(trade, 0, "price")
        exit_price = self._safe_history_value(trade, -1, "price")
        size = self._safe_history_value(trade, 0, "size")

        return_pct = None
        if entry_price not in (None, 0.0) and exit_price is not None:
            return_pct = ((float(exit_price) / float(entry_price)) - 1.0) * 100.0

        self.trades.append(
            {
                "symbol": symbol,
                "entry_time": pd.Timestamp(entry_dt),
                "exit_time": pd.Timestamp(exit_dt),
                "entry_date": entry_dt.date().isoformat(),
                "exit_date": exit_dt.date().isoformat(),
                "size": size,
                "entry_price": entry_price,
                "exit_price": exit_price,
                "pnl": float(trade.pnlcomm),
                "return_pct": return_pct,
                "exit_reason": getattr(self.strategy, "last_exit_reason", None),
            }
        )

    def get_analysis(self):
        return self.trades


def _daily_to_bt_frame(daily_df: pd.DataFrame) -> pd.DataFrame:
    """Convert normalized daily frame to Backtrader OHLCV names."""
    return daily_df.rename(
        columns={
            "open": "Open",
            "high": "High",
            "low": "Low",
            "close": "Close",
            "volume": "Volume",
        }
    )[["Open", "High", "Low", "Close", "Volume"]].copy()


def _run_symbol_backtest(
    *,
    symbol: str,
    symbols_data: dict,
    from_date: str,
    to_date: str,
    importer_name: str,
    initial_cash: float,
    commission: float,
    trade_size: int,
    ma_period: int,
    atr_period: int,
    gap_up_threshold: float,
    printlog: bool,
    intraday_df_preloaded: pd.DataFrame | None = None,
):
    """Internal single-symbol backtest utility used by both single/multi runners."""
    daily_df = prepare_daily_frame(symbols_data, symbol)
    daily_context = build_daily_context(
        daily_df,
        ma_period=ma_period,
        atr_period=atr_period,
        gap_up_threshold=gap_up_threshold,
    )

    if intraday_df_preloaded is not None:
        intraday_df = intraday_df_preloaded.copy()
    else:
        intraday_df = load_intraday_3m_frame(
            symbol=symbol,
            from_date=from_date,
            to_date=to_date,
            importer_name=importer_name,
            resample="3min",
        )

    # Keep only intraday bars that have available daily context for same-day checks.
    valid_days = set(daily_context.keys())
    intraday_df = intraday_df.loc[pd.Index(intraday_df.index.date).isin(valid_days)].copy()
    if intraday_df.empty:
        raise ValueError("No intraday bars remain after aligning with daily context dates")

    daily_bt = _daily_to_bt_frame(daily_df)

    cerebro = bt.Cerebro(stdstats=False)
    cerebro.broker.setcash(float(initial_cash))
    cerebro.broker.setcommission(commission=float(commission))

    # Multi-timeframe requirement: data0 intraday, data1 daily.
    cerebro.adddata(IntradayVWAPData(dataname=intraday_df), name=symbol)
    cerebro.adddata(bt.feeds.PandasData(dataname=daily_bt), name=f"{symbol}_daily")

    cerebro.addstrategy(
        EpisodicPivot20DMAOverlapStrategy,
        daily_context_by_date=daily_context,
        trade_size=int(trade_size),
        printlog=bool(printlog),
    )
    cerebro.addanalyzer(TradeCaptureAnalyzer, _name="trade_capture")

    start_value = float(cerebro.broker.getvalue())
    result = cerebro.run()[0]
    end_value = float(cerebro.broker.getvalue())

    trades = pd.DataFrame(result.analyzers.trade_capture.get_analysis())
    if not trades.empty:
        trades = trades.sort_values(["entry_time", "symbol"]).reset_index(drop=True)

    return {
        "strategy": result,
        "cerebro": cerebro,
        "symbol": symbol,
        "rows_daily": int(len(daily_df)),
        "rows_intraday": int(len(intraday_df)),
        "start_value": start_value,
        "end_value": end_value,
        "net_pnl": end_value - start_value,
        "daily_context": daily_context,
        "intraday_df": intraday_df,
        "daily_df": daily_df,
        "trades": trades,
    }


def run_episodic_pivot_20dma_overlap_for_symbol(
    *,
    symbol: str,
    from_date: str,
    to_date: str,
    daily_variables_filename: str = "daily_variables.pkl",
    importer_name: str = "intraday_import",
    initial_cash: float = 100_000.0,
    commission: float = 0.001,
    trade_size: int = 1,
    ma_period: int = 20,
    atr_period: int = 14,
    gap_up_threshold: float = 0.08,
    printlog: bool = False,
    symbols_data: dict | None = None,
):
    """Single-symbol entrypoint kept for convenience."""
    loaded_symbols = symbols_data
    if loaded_symbols is None:
        loaded_symbols = load_daily_variables(filename=daily_variables_filename)
        if not isinstance(loaded_symbols, dict):
            raise TypeError(f"Expected `symbols` to be dict, got {type(loaded_symbols)!r}")

    return _run_symbol_backtest(
        symbol=symbol,
        symbols_data=loaded_symbols,
        from_date=from_date,
        to_date=to_date,
        importer_name=importer_name,
        initial_cash=float(initial_cash),
        commission=float(commission),
        trade_size=int(trade_size),
        ma_period=int(ma_period),
        atr_period=int(atr_period),
        gap_up_threshold=float(gap_up_threshold),
        printlog=bool(printlog),
    )


def run_episodic_pivot_20dma_overlap_multi_symbol(
    *,
    symbol_list: list[str],
    from_date: str,
    to_date: str,
    daily_variables_filename: str = "daily_variables.pkl",
    importer_name: str = "intraday_import",
    initial_cash_per_symbol: float = 100_000.0,
    commission: float = 0.001,
    trade_size: int = 1,
    ma_period: int = 20,
    atr_period: int = 14,
    gap_up_threshold: float = 0.08,
    printlog: bool = False,
):
    """Run the episodic strategy across multiple symbols and aggregate outputs.

    Output tables are designed to resemble `example_multi_symbol.ipynb`:
    - portfolio summary (`summary`)
    - per-symbol trade summary (`symbol_trade_summary`)
    - sample closed trades (`trades`)
    """
    if not symbol_list:
        raise ValueError("`symbol_list` must contain at least one symbol")

    symbols_data = load_daily_variables(filename=daily_variables_filename)
    if not isinstance(symbols_data, dict):
        raise TypeError(f"Expected `symbols` to be dict, got {type(symbols_data)!r}")

    selected_symbols = [str(s).upper() for s in symbol_list]

    # One batched importer call for all symbols (lets price_data_import handle concurrency).
    intraday_frames_by_symbol = load_intraday_3m_frames_batch(
        symbol_list=selected_symbols,
        from_date=from_date,
        to_date=to_date,
        importer_name=importer_name,
        resample="3min",
    )

    run_rows: list[dict] = []
    trade_frames: list[pd.DataFrame] = []
    outputs_by_symbol: dict[str, dict] = {}

    for sym in selected_symbols:
        try:
            intraday_preloaded = intraday_frames_by_symbol.get(sym)
            if intraday_preloaded is None:
                raise ValueError(f"No intraday data returned for {sym} in batch import")

            out = _run_symbol_backtest(
                symbol=sym,
                symbols_data=symbols_data,
                from_date=from_date,
                to_date=to_date,
                importer_name=importer_name,
                initial_cash=float(initial_cash_per_symbol),
                commission=float(commission),
                trade_size=int(trade_size),
                ma_period=int(ma_period),
                atr_period=int(atr_period),
                gap_up_threshold=float(gap_up_threshold),
                printlog=bool(printlog),
                intraday_df_preloaded=intraday_preloaded,
            )

            outputs_by_symbol[sym] = out
            trades = out["trades"]
            if not trades.empty:
                trade_frames.append(trades)

            run_rows.append(
                {
                    "symbol": sym,
                    "status": "ok",
                    "rows_daily": out["rows_daily"],
                    "rows_intraday": out["rows_intraday"],
                    "trades": int(len(trades)),
                    "start_value": float(out["start_value"]),
                    "end_value": float(out["end_value"]),
                    "net_pnl": float(out["net_pnl"]),
                }
            )
        except Exception as exc:
            run_rows.append(
                {
                    "symbol": sym,
                    "status": "error",
                    "rows_daily": None,
                    "rows_intraday": None,
                    "trades": 0,
                    "start_value": None,
                    "end_value": None,
                    "net_pnl": None,
                    "reason": str(exc),
                }
            )

    run_df = pd.DataFrame(run_rows).sort_values(["status", "symbol"]).reset_index(drop=True)

    if trade_frames:
        trades = pd.concat(trade_frames, ignore_index=True).sort_values(["entry_time", "symbol"]).reset_index(drop=True)
    else:
        trades = pd.DataFrame(
            columns=[
                "symbol",
                "entry_time",
                "exit_time",
                "entry_date",
                "exit_date",
                "size",
                "entry_price",
                "exit_price",
                "pnl",
                "return_pct",
                "exit_reason",
            ]
        )

    ok_df = run_df[run_df["status"] == "ok"].copy()
    total_initial = float(ok_df["start_value"].sum()) if not ok_df.empty else 0.0
    total_final = float(ok_df["end_value"].sum()) if not ok_df.empty else 0.0
    total_pnl = total_final - total_initial
    total_return_pct = (total_pnl / total_initial * 100.0) if total_initial > 0 else 0.0

    summary = pd.DataFrame(
        {
            "metric": [
                "symbol_count",
                "symbols_ok",
                "symbols_error",
                "strategy_initial_total",
                "strategy_final_total",
                "strategy_net_pnl",
                "strategy_return_%",
                "num_closed_trades",
            ],
            "value": [
                float(len(selected_symbols)),
                float((run_df["status"] == "ok").sum()),
                float((run_df["status"] != "ok").sum()),
                float(total_initial),
                float(total_final),
                float(total_pnl),
                float(total_return_pct),
                float(len(trades)),
            ],
        }
    )

    if not trades.empty:
        symbol_trade_summary = (
            trades.groupby("symbol", as_index=False)
            .agg(
                num_trades=("pnl", "count"),
                wins=("pnl", lambda x: int((x > 0).sum())),
                total_pnl=("pnl", "sum"),
                avg_pnl=("pnl", "mean"),
                avg_return_pct=("return_pct", "mean"),
            )
            .sort_values("total_pnl", ascending=False)
            .reset_index(drop=True)
        )
        symbol_trade_summary["win_rate"] = (
            symbol_trade_summary["wins"] / symbol_trade_summary["num_trades"] * 100.0
        )
    else:
        symbol_trade_summary = pd.DataFrame(
            columns=["symbol", "num_trades", "wins", "total_pnl", "avg_pnl", "avg_return_pct", "win_rate"]
        )

    return {
        "selected_symbols": selected_symbols,
        "outputs_by_symbol": outputs_by_symbol,
        "run_df": run_df,
        "trades": trades,
        "summary": summary,
        "symbol_trade_summary": symbol_trade_summary,
    }


def display_episodic_multi_symbol_results(output: dict, sample_trades: int = 20) -> None:
    """Display multi-symbol outputs in an example_multi_symbol-like layout."""
    print("=== Portfolio Summary ===")
    display(output["summary"])

    print("=== Per-Symbol Trade Summary ===")
    display(output["symbol_trade_summary"])

    print("=== Sample Closed Trades ===")
    display(output["trades"].head(int(sample_trades)))

    print("=== Run Status by Symbol ===")
    display(output["run_df"])


In [3]:
def _normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    return df.rename(columns={c: str(c).strip().lower().replace(" ", "_") for c in df.columns})


def _ensure_datetime_index(df: pd.DataFrame, *, frame_name: str) -> pd.DataFrame:
    out = df.copy()
    if not isinstance(out.index, pd.DatetimeIndex):
        for candidate in ("date", "datetime", "timestamp"):
            if candidate in out.columns:
                out[candidate] = pd.to_datetime(out[candidate], errors="coerce")
                out = out.set_index(candidate)
                break

    if not isinstance(out.index, pd.DatetimeIndex):
        raise TypeError(f"{frame_name}: expected DatetimeIndex or date/datetime/timestamp column")

    out.index = pd.to_datetime(out.index, errors="coerce")
    if getattr(out.index, "tz", None) is not None:
        out.index = out.index.tz_convert(None)

    return out.sort_index()


def _pick_first_present(df: pd.DataFrame, candidates: list[str]) -> str | None:
    for c in candidates:
        if c in df.columns:
            return c
    return None


def _compute_atr(df: pd.DataFrame, period: int = 14) -> pd.Series:
    prev_close = df["close"].shift(1)
    tr_components = pd.concat(
        [
            (df["high"] - df["low"]).abs(),
            (df["high"] - prev_close).abs(),
            (df["low"] - prev_close).abs(),
        ],
        axis=1,
    )
    tr = tr_components.max(axis=1)
    return tr.rolling(period, min_periods=period).mean()


def prepare_daily_frame(symbols: dict, symbol: str) -> pd.DataFrame:
    """Read a symbol's inter-day dataframe from `symbols[symbol].df` and normalize."""
    if symbol not in symbols:
        raise KeyError(f"{symbol} not found in daily symbols dictionary")

    symbol_data = symbols[symbol]
    if not hasattr(symbol_data, "df"):
        raise TypeError(f"symbols[{symbol!r}] has no `.df` attribute")

    df = _normalize_columns(symbol_data.df.copy())
    df = _ensure_datetime_index(df, frame_name=f"daily {symbol}")

    for col in ("open", "high", "low", "close"):
        if col not in df.columns:
            raise ValueError(f"daily {symbol}: missing required column `{col}`")

    if "volume" not in df.columns:
        df["volume"] = 0.0

    for col in ("open", "high", "low", "close", "volume"):
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df.dropna(subset=["open", "high", "low", "close"]).copy()


def build_daily_context(
    daily_df: pd.DataFrame,
    *,
    ma_period: int = 20,
    atr_period: int = 14,
    gap_up_threshold: float = 0.08,
) -> dict:
    """Build day-level context map used by the intraday strategy.

    Why precompute context:
    - Your entry requires a daily condition and a 3-minute condition on the same day.
    - Precomputing daily state by date lets us align same-day daily + intraday logic
      cleanly in backtrader without adding lookahead in indicator calculations.
    """
    df = daily_df.copy()

    ma_col = _pick_first_present(df, ["20dma", "sma20", "sma_20", "ma20", "ma_20"])
    if ma_col is None:
        df["_ma20_calc"] = df["close"].rolling(ma_period, min_periods=ma_period).mean()
        ma_col = "_ma20_calc"

    atr_col = _pick_first_present(df, ["atr_14", "atr14", "atr", "atr_20"])
    if atr_col is None:
        df["_atr_calc"] = _compute_atr(df, period=atr_period)
        atr_col = "_atr_calc"

    ma20 = pd.to_numeric(df[ma_col], errors="coerce")
    atr = pd.to_numeric(df[atr_col], errors="coerce")

    prev_close = pd.to_numeric(df["close"], errors="coerce").shift(1)
    gap_up = (pd.to_numeric(df["open"], errors="coerce") / prev_close - 1.0) >= float(gap_up_threshold)
    close_above_ma = pd.to_numeric(df["close"], errors="coerce") > ma20
    overlap_20dma = (pd.to_numeric(df["high"], errors="coerce") > ma20) & (pd.to_numeric(df["low"], errors="coerce") < ma20)

    # Flag lifecycle:
    # - turns on when gap-up + close > 20DMA is true
    # - remains active while close > 20DMA
    flag_active = []
    is_active = False
    for trigger, still_above in zip(gap_up.fillna(False), close_above_ma.fillna(False)):
        if bool(trigger):
            is_active = True
        if is_active and (not bool(still_above)):
            is_active = False
        flag_active.append(bool(is_active))

    close_below_ma = pd.to_numeric(df["close"], errors="coerce") < ma20
    atr_floor = ma20 - (2.0 * atr)

    context = {}
    for ts, fa, ov, cbm, ma_val, atr_val, floor_val in zip(
        df.index,
        flag_active,
        overlap_20dma,
        close_below_ma,
        ma20,
        atr,
        atr_floor,
    ):
        if pd.isna(ma_val) or pd.isna(atr_val) or pd.isna(floor_val):
            continue
        context[pd.Timestamp(ts).date()] = {
            "flag_active": bool(fa),
            "overlap_20dma": bool(ov),
            "close_below_20dma": bool(cbm),
            "ma20": float(ma_val),
            "atr": float(atr_val),
            "atr_floor": float(floor_val),
        }

    return context


def _normalize_intraday_raw_frame(df_raw: pd.DataFrame, symbol: str) -> pd.DataFrame:
    """Normalize one raw intraday frame into Backtrader-ready OHLCV+VWAP columns."""
    df = _normalize_columns(df_raw.copy())
    df = _ensure_datetime_index(df, frame_name=f"intraday {symbol}")

    for col in ("open", "high", "low", "close", "volume", "vwap"):
        if col not in df.columns:
            raise ValueError(f"intraday {symbol}: missing required column `{col}`")
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["open", "high", "low", "close", "volume", "vwap"]).copy()

    return df.rename(
        columns={
            "open": "Open",
            "high": "High",
            "low": "Low",
            "close": "Close",
            "volume": "Volume",
            "vwap": "VWAP",
        }
    )[["Open", "High", "Low", "Close", "Volume", "VWAP"]]


def load_intraday_3m_frames_batch(
    *,
    symbol_list: list[str],
    from_date: str,
    to_date: str,
    importer_name: str = "intraday_import",
    resample: str = "3min",
    timespan: str = "minute",
    multiplier: int = 1,
    limit: int = 50_000,
    market_open_only: bool = True,
    importer_kwargs: dict | None = None,
) -> dict[str, pd.DataFrame]:
    """Batch-load intraday data for many symbols in one importer call.

    This intentionally uses a single `price_data_import` call with all symbols so the
    importer can use its own internal concurrency efficiently.
    """
    if importer_name not in INTRADAY_IMPORTERS:
        raise ValueError(f"Unknown importer `{importer_name}`. Available: {sorted(INTRADAY_IMPORTERS)}")
    if not symbol_list:
        return {}

    importer = INTRADAY_IMPORTERS[importer_name]

    kwargs = {
        "wl": [str(s).upper() for s in symbol_list],
        "from_date": from_date,
        "to_date": to_date,
        "resample": resample,
        "timespan": timespan,
        "multiplier": multiplier,
        "limit": int(limit),
        "market_open_only": market_open_only,
    }
    if importer_kwargs:
        kwargs.update(importer_kwargs)

    pulled = importer(**kwargs)
    normalized: dict[str, pd.DataFrame] = {}

    for sym in kwargs["wl"]:
        if sym not in pulled or pulled[sym] is None or len(pulled[sym]) == 0:
            continue
        normalized[sym] = _normalize_intraday_raw_frame(pulled[sym], sym)

    return normalized


def load_intraday_3m_frame(
    *,
    symbol: str,
    from_date: str,
    to_date: str,
    importer_name: str = "intraday_import",
    resample: str = "3min",
    timespan: str = "minute",
    multiplier: int = 1,
    limit: int = 50_000,
    market_open_only: bool = True,
    importer_kwargs: dict | None = None,
) -> pd.DataFrame:
    """Single-symbol wrapper around the batch intraday loader."""
    frames = load_intraday_3m_frames_batch(
        symbol_list=[symbol],
        from_date=from_date,
        to_date=to_date,
        importer_name=importer_name,
        resample=resample,
        timespan=timespan,
        multiplier=multiplier,
        limit=limit,
        market_open_only=market_open_only,
        importer_kwargs=importer_kwargs,
    )
    key = str(symbol).upper()
    if key not in frames:
        raise ValueError(f"No intraday data returned for {symbol}")
    return frames[key]


class IntradayVWAPData(bt.feeds.PandasData):
    """3-minute feed with VWAP line."""

    lines = ("vwap",)
    params = (
        ("datetime", None),
        ("open", "Open"),
        ("high", "High"),
        ("low", "Low"),
        ("close", "Close"),
        ("volume", "Volume"),
        ("openinterest", -1),
        ("vwap", "VWAP"),
    )


class EpisodicPivot20DMAOverlapStrategy(bt.Strategy):
    """Entry = daily regime + daily overlap + intraday trigger. Exit/stop per prompt."""

    params = dict(
        daily_context_by_date=None,
        trade_size=1,
        allow_multiple_entries_per_day=False,
        printlog=False,
    )

    def __init__(self) -> None:
        # data0: 3-minute feed, data1: daily feed (for strict multi-timeframe setup)
        self.intra = self.datas[0]
        self.daily = self.datas[1] if len(self.datas) > 1 else None

        # Session-reset intraday EMA state (resets at each new day).
        # This prevents EMA values from carrying over across sessions.
        self._ema5_alpha = 2.0 / (5.0 + 1.0)
        self._ema9_alpha = 2.0 / (9.0 + 1.0)
        self._ema_session_day = None
        self._ema5_session = None
        self._ema9_session = None
        self._prev_diff_session = None
        self._cross_up_session = False

        # Map: date -> dict(flag_active, overlap_20dma, close_below_20dma, ma20, atr, atr_floor)
        self.daily_context_by_date = {
            pd.Timestamp(k).date(): v for k, v in (self.p.daily_context_by_date or {}).items()
        }

        self.pending_order = None
        self.current_day = None
        self.running_day_low = None

        self.entry_day = None
        self.entry_day_stop = None
        self.last_entry_day = None
        self.last_exit_reason = None

    def log(self, message: str) -> None:
        if self.p.printlog:
            dt = self.intra.datetime.datetime(0)
            print(f"{dt.isoformat()} | {message}")

    def notify_order(self, order) -> None:
        if order.status in [order.Submitted, order.Accepted]:
            return

        if order.status in [order.Canceled, order.Margin, order.Rejected]:
            self.pending_order = None
            return

        if order.status == order.Completed:
            if order.isbuy():
                self.entry_day = self.intra.datetime.date(0)
                self.last_entry_day = self.entry_day
                self.log(
                    f"BUY filled @ {order.executed.price:.2f} | stop(entry-day-low)={self.entry_day_stop:.2f}"
                )
            else:
                self.log(f"SELL filled @ {order.executed.price:.2f} | reason={self.last_exit_reason}")
            self.pending_order = None

    def notify_trade(self, trade) -> None:
        if trade.isclosed:
            self.entry_day = None
            self.entry_day_stop = None

    def _update_running_day_low(self) -> None:
        bar_day = self.intra.datetime.date(0)
        bar_low = float(self.intra.low[0])

        if self.current_day != bar_day:
            self.current_day = bar_day
            self.running_day_low = bar_low
        else:
            self.running_day_low = min(float(self.running_day_low), bar_low)

    def _update_session_ema_cross(self) -> None:
        """Update 5/9 EMA values that reset every new trading day."""
        bar_day = self.intra.datetime.date(0)
        close_ = float(self.intra.close[0])

        # First bar of a new day: initialize from current close.
        if self._ema_session_day != bar_day:
            self._ema_session_day = bar_day
            self._ema5_session = close_
            self._ema9_session = close_
            self._prev_diff_session = 0.0
            self._cross_up_session = False
            return

        self._ema5_session = (self._ema5_alpha * close_) + ((1.0 - self._ema5_alpha) * float(self._ema5_session))
        self._ema9_session = (self._ema9_alpha * close_) + ((1.0 - self._ema9_alpha) * float(self._ema9_session))

        diff = float(self._ema5_session) - float(self._ema9_session)
        prev = self._prev_diff_session
        self._cross_up_session = bool((prev is not None) and (prev <= 0.0) and (diff > 0.0))
        self._prev_diff_session = diff

    def _entry_allowed_today(self, bar_day) -> bool:
        if self.p.allow_multiple_entries_per_day:
            return True
        return self.last_entry_day != bar_day

    def next(self) -> None:
        self._update_running_day_low()
        self._update_session_ema_cross()

        if self.pending_order is not None:
            return

        bar_day = self.intra.datetime.date(0)
        ctx = self.daily_context_by_date.get(bar_day)
        if ctx is None:
            return

        close_ = float(self.intra.close[0])
        low_ = float(self.intra.low[0])
        vwap_ = float(self.intra.vwap[0])

        # Entry block
        if not self.position:
            if not bool(ctx["flag_active"]):
                return
            if not bool(ctx["overlap_20dma"]):
                return

            # Intraday trigger uses session-reset EMA cross (5 above 9) + close below VWAP.
            intraday_trigger = (close_ < vwap_) and bool(self._cross_up_session)
            if intraday_trigger and self._entry_allowed_today(bar_day):
                # Stop is low of entry day, captured as day-low-so-far at entry time.
                self.entry_day_stop = float(self.running_day_low)
                self.last_exit_reason = None
                self.pending_order = self.buy(size=int(self.p.trade_size))
                self.log(
                    f"ENTRY signal | close={close_:.2f} < vwap={vwap_:.2f} and EMA5 crossed above EMA9"
                )
            return

        # Stop signal: low of day of entry.
        if self.entry_day_stop is not None and low_ <= float(self.entry_day_stop):
            self.last_exit_reason = "stop_low_of_entry_day"
            self.pending_order = self.close()
            return

        # Exit signal 1: daily close < 20DMA.
        if bool(ctx["close_below_20dma"]):
            self.last_exit_reason = "daily_close_below_20dma"
            self.pending_order = self.close()
            return

        # Exit signal 2: 3m close < (20DMA - 2*ATR).
        if close_ < float(ctx["atr_floor"]):
            self.last_exit_reason = "intraday_close_below_20dma_minus_2atr"
            self.pending_order = self.close()


def run_episodic_pivot_20dma_overlap_for_symbol(
    *,
    symbol: str,
    from_date: str,
    to_date: str,
    daily_variables_filename: str = "daily_variables.pkl",
    importer_name: str = "intraday_import",
    initial_cash: float = 100_000.0,
    commission: float = 0.001,
    trade_size: int = 1,
    ma_period: int = 20,
    atr_period: int = 14,
    gap_up_threshold: float = 0.08,
    printlog: bool = False,
):
    """Load data using AGENTS.md conventions, then run the Backtrader strategy."""
    # Inter-day data from strategies.data loaders.
    symbols = load_daily_variables(filename=daily_variables_filename)
    if not isinstance(symbols, dict):
        raise TypeError(f"Expected `symbols` to be dict, got {type(symbols)!r}")

    daily_df = prepare_daily_frame(symbols, symbol)
    daily_context = build_daily_context(
        daily_df,
        ma_period=ma_period,
        atr_period=atr_period,
        gap_up_threshold=gap_up_threshold,
    )

    intraday_df = load_intraday_3m_frame(
        symbol=symbol,
        from_date=from_date,
        to_date=to_date,
        importer_name=importer_name,
        resample="3min",
    )

    # Restrict intraday bars to dates where daily context is available.
    valid_days = set(daily_context.keys())
    intraday_df = intraday_df.loc[pd.Index(intraday_df.index.date).isin(valid_days)].copy()
    if intraday_df.empty:
        raise ValueError("No intraday bars remain after aligning with daily context dates")

    cerebro = bt.Cerebro(stdstats=False)
    cerebro.broker.setcash(float(initial_cash))
    cerebro.broker.setcommission(commission=float(commission))

    # Keep both timeframes in the Backtrader engine:
    # - data0: 3-minute bars used for trigger/stop timing
    # - data1: daily bars included for strict multi-timeframe setup
    daily_bt = daily_df.rename(
        columns={
            "open": "Open",
            "high": "High",
            "low": "Low",
            "close": "Close",
            "volume": "Volume",
        }
    )[["Open", "High", "Low", "Close", "Volume"]].copy()

    cerebro.adddata(IntradayVWAPData(dataname=intraday_df), name=f"{symbol}_3m")
    cerebro.adddata(bt.feeds.PandasData(dataname=daily_bt), name=f"{symbol}_daily")

    cerebro.addstrategy(
        EpisodicPivot20DMAOverlapStrategy,
        daily_context_by_date=daily_context,
        trade_size=int(trade_size),
        printlog=bool(printlog),
    )

    start_value = float(cerebro.broker.getvalue())
    result = cerebro.run()[0]
    end_value = float(cerebro.broker.getvalue())

    return {
        "strategy": result,
        "cerebro": cerebro,
        "symbol": symbol,
        "rows_daily": int(len(daily_df)),
        "rows_intraday": int(len(intraday_df)),
        "start_value": start_value,
        "end_value": end_value,
        "net_pnl": end_value - start_value,
    }

In [5]:
# ------------------------------------------
# Example (edit and run): Multi-symbol batch
# ------------------------------------------
# SELECTED_SYMBOLS = [
#     "VRT",
#     "NVDA",
#     "AAPL",
#     "MSFT",
#     "META",
# ]
SELECTED_SYMBOLS = [line.replace("\n", "") for line in open(r"E:\Market Research\Studies\Sector Studies\Watchlists\High_AvgDV.txt").readlines()]

multi_output = run_episodic_pivot_20dma_overlap_multi_symbol(
    symbol_list=SELECTED_SYMBOLS,
    from_date="2024-01-01",
    to_date="2026-02-16",
    importer_name="intraday_import",  # or nonconsecutive_intraday_import / fragmented_intraday_import
    trade_size=1,
    printlog=False,
)

display_episodic_multi_symbol_results(multi_output, sample_trades=20)

Importing Price Data: 100%|██████████| 94/94 [08:07<00:00,  5.19s/it]  


=== Portfolio Summary ===


,metric,value
0,symbol_count,9.400000e+01
1,symbols_ok,9.400000e+01
2,symbols_error,0.000000e+00
3,strategy_initial_total,9.400000e+06
4,strategy_final_total,9.400109e+06
5,strategy_net_pnl,1.090467e+02
6,strategy_return_%,1.160071e-03
7,num_closed_trades,1.030000e+02


=== Per-Symbol Trade Summary ===


,symbol,num_trades,wins,total_pnl,avg_pnl,avg_return_pct,win_rate
0,LITE,4,2,36.914730,9.228682,NaN,50.0
1,VRT,4,2,24.971368,6.242842,NaN,50.0
2,ZBRA,4,2,17.643485,4.410871,NaN,50.0
3,ALGN,4,2,17.150655,4.287664,NaN,50.0
4,ANET,2,2,16.824319,8.412159,NaN,100.0
5,SITM,2,1,11.183405,5.591702,NaN,50.0
6,UNF,1,1,8.075060,8.075060,NaN,100.0
7,NKTR,5,2,5.377283,1.075457,NaN,40.0
8,RNA,4,2,3.897925,0.974481,NaN,50.0
9,SEI,2,2,3.606852,1.803426,NaN,100.0


=== Sample Closed Trades ===


,symbol,entry_time,exit_time,entry_date,exit_date,size,entry_price,exit_price,pnl,return_pct,exit_reason
0,JHX,2024-01-09 11:27:00,2024-01-17 09:33:00,2024-01-09,2024-01-17,None,None,None,-0.143090,None,daily_close_below_20dma
1,CROX,2024-01-17 12:51:00,2024-02-02 09:33:00,2024-01-17,2024-02-02,None,None,None,-1.283155,None,daily_close_below_20dma
2,RNA,2024-01-22 09:36:00,2024-01-23 09:33:00,2024-01-22,2024-01-23,None,None,None,0.294455,None,daily_close_below_20dma
3,ALGN,2024-02-01 14:00:00,2024-02-02 09:48:00,2024-02-01,2024-02-02,None,None,None,-11.357895,None,stop_low_of_entry_day
4,ALGN,2024-02-05 10:33:00,2024-03-01 09:33:00,2024-02-05,2024-03-01,None,None,None,28.054495,None,daily_close_below_20dma
5,VIAV,2024-02-12 10:36:00,2024-02-13 09:33:00,2024-02-12,2024-02-13,None,None,None,-0.409810,None,daily_close_below_20dma
6,ENPH,2024-03-05 13:12:00,2024-03-12 09:33:00,2024-03-05,2024-03-12,None,None,None,0.686680,None,daily_close_below_20dma
7,GLDD,2024-03-08 10:45:00,2024-03-08 11:12:00,2024-03-08,2024-03-08,None,None,None,-0.062615,None,stop_low_of_entry_day
8,XPO,2024-03-11 12:24:00,2024-03-19 09:33:00,2024-03-11,2024-03-19,None,None,None,0.209390,None,daily_close_below_20dma
9,ZBRA,2024-03-14 10:15:00,2024-03-14 12:15:00,2024-03-14,2024-03-14,None,None,None,-3.423960,None,stop_low_of_entry_day


=== Run Status by Symbol ===


,symbol,status,rows_daily,rows_intraday,trades,start_value,end_value,net_pnl
0,AAP,ok,2513,68764,0,100000.0,100000.000000,0.000000
1,ADNT,ok,2335,68251,0,100000.0,100000.000000,0.000000
2,AEHR,ok,2513,67924,2,100000.0,99992.769511,-7.230489
3,ALGN,ok,2513,67375,4,100000.0,100017.150655,17.150655
4,AMAT,ok,2513,68805,0,100000.0,100000.000000,0.000000
...,...,...,...,...,...,...,...,...
89,VSTS,ok,595,68336,0,100000.0,100000.000000,0.000000
90,WS,ok,552,50822,0,100000.0,100000.000000,0.000000
91,WWD,ok,2513,61917,0,100000.0,100000.000000,0.000000
92,XPO,ok,2513,68380,1,100000.0,100000.209390,0.209390
